In [461]:
import pandas as pd
import re

data = pd.read_csv('urban_data.csv')
print(data.columns)

# Remove rows with reference links

print('Before: ', data['Header'].unique().tolist())
remove = ['Link', 'Document', 'Remarks', 'Source']
pattern = '|'.join(re.escape(word) for word in remove)
data = data[~data['Header'].str.contains(pattern)]
print('After: ', data['Header'].unique().tolist())

Index(['ATO UC Code', 'Urban Center', 'Indicator code', 'Indicator name',
       'Indicator source', 'Units', 'Header', 'Year', 'Value'],
      dtype='object')
Before:  ['Series', '[1]', 'Document [1]', 'Source [1]', 'Link [1]', 'Remarks [1]', '[2]', 'Document [2]', 'Link [2]', 'Source [2]', '[3]', 'Document [3]', 'Link [3]', 'Remarks [2]', 'Source [3]', '[4]', 'Document [4]', 'Link [4]', 'Remarks', 'Remarks [3]', 'Source [4]', 'Document [5]', 'Link [5]', 'Document [6]', 'Link [6]', '[6]', 'Rank', 'Economics', 'Human Capital', 'Quality of Life', 'Environment', 'Governance']
After:  ['Series', '[1]', '[2]', '[3]', '[4]', '[6]', 'Rank', 'Economics', 'Human Capital', 'Quality of Life', 'Environment', 'Governance']


In [462]:
# Convert string to int for year, string to float for value

print(data['Year'].unique())
data['Year'] = data['Year'].astype(int)

print(data['Value'].unique().tolist())
data['Value'] = data['Value'].astype(float)

['2017' '2023' '2018' '2019' '2020' '2022' '2016' '2021' '2024']
['3650000', '3955000', '4070000', '3250000', '16235000', '20390000', '10680000', '7440000', '18760000', '22685000', '12240000', '4715000', '7280000', '10165000', '9985000', '25735000', '14810000', '22885000', '31320000', '13670000', '2645000', '16985000', '2395000', '37750000', '7365000', '1370000', '10355000', '22930000', '3885000', '23575000', '16570000', '5140000', '5725000', '2540000', '15315000', '7445000', '10075000', '14280000', '20605000', '6550000', '9520000', '15910000', '10870000', '4085000', '6790000', '5270000', '20230000', '1635000', '12830000', '2685000', '5845000', '8655000', '6240000', '13520000', '3630000', '10350000', '15135000', '20685000', '4273156', '4529500', '4840600', '7000000', '18627000', '18522000', '14645000', '12135000', '26940000', '24073000', '17619000', '3707090', '7450000', '15386000', '1055450', '12395000', '32226000', '18502000', '24973000', '33756000', '14148000', '2286000', '1464890',

In [463]:
# subset indicator data based on filters (determined based on year, source with highest data completeness)
def load_indicator(data, indicator, year=None, source=None, header=None):
    mask = (data['Indicator name'] == indicator)
    cols = ['Urban Center', 'Year', 'Value', 'Header', 'Indicator source', 'Units']

    if year:
        mask &= (data['Year'] ==year)
    if source:
        mask &= (data['Indicator source'] == source)
    if header:
        mask &= (data['Header'] == header)
        
    return data.loc[mask, cols]

# prints summary (indicator, source, header, year, count) of indicator
def indicator_source_year_coverage(data, indicator):
    subset = data[data['Indicator name']==indicator]
    return (
        subset
        .groupby(['Indicator name', 'Indicator source', 'Header', 'Year'])
        .size()
        .reset_index(name='count')
        .sort_values(['Indicator name', 'count', 'Year', 'Indicator source', 'Header'], 
                     ascending=[True, False, False, True, True])
    )

# from summary of indicator_source_year_coverage, find best year + source (based on highest count)
def best_year_source(summary):
    if summary.iloc[0]['count'] > 30:
        return summary.iloc[0]['Year'], summary.iloc[0]['Indicator source'], summary.iloc[0]['Header'], summary.iloc[0]['count']
    return None, None, None, summary.iloc[0]['count']

def remove_duplicate(data):
    data = data.groupby('Urban Center', as_index=False)['Value'].mean()
    return data


In [464]:
## Cross sectional data with a pivot table 

# 1. Check data completeness

# append to list of dictionary {year, source, header}
indicator_list = []
print(data['Indicator name'].unique())

['Population' 'Area' 'Population Density' 'Cost of Living Index'
 'Built-up areas per capita' 'Transport landuse %'
 'Road Length per capita' 'Pedestrian Streets per capita'
 'Rapid Transit to Resident Ratio (RTR)'
 'Modal split (Passenger) - by trips - Car'
 'Modal split (Passenger) - by trips - Public Transport'
 'Congestion Ranking' 'Traffic Index' 'Avg. trip length (Car)'
 'Avg. trip length (Bus)' 'Avg. trip length (Public Transport)'
 'Average travel speed' 'Percentage Access to Public Transport'
 'Share of population with access to open public spaces'
 'Road traffic crash (Total)' 'Surface transport SOx Concentration'
 'Surface transport NOx Concentration' 'Pollutant concentration (PM 10)'
 'Oxford Economics Global Cities Index']


In [465]:
master_table = pd.DataFrame({'Urban Center': data['Urban Center'].unique()})

for indicator in data['Indicator name'].unique():
    indicator_summary = indicator_source_year_coverage(data, indicator)

    if indicator == 'Oxford Economics Global Cities Index':
        headers = data[data['Indicator name']==indicator]['Header'].unique()
        for header in headers:
            col_name = indicator
            current = load_indicator(data, indicator=indicator, year=2024, source=None, header=header)
            current = current[['Urban Center', 'Value']]
            col_name += f" ({header})"
            current.rename(columns={'Value': col_name}, inplace=True)
            master_table = master_table.merge(current, on='Urban Center', how='left')

    else:
        year, source, header, count = best_year_source(indicator_summary)
        col_name = indicator
        current = load_indicator(data, indicator=indicator, year=year, source=source, header=header)
        current = current[['Urban Center', 'Value']]
        current = remove_duplicate(current)
        if not year:
            continue
        
        if year < 2021:
            col_name += f" ({year})"

        print(indicator, count)
        current.rename(columns={'Value': col_name}, inplace=True)
        master_table = master_table.merge(current, on='Urban Center', how='left')


master_table.set_index('Urban Center', inplace=True)

Population 63
Area 60
Population Density 62
Cost of Living Index 49
Built-up areas per capita 62
Road Length per capita 52
Pedestrian Streets per capita 54
Rapid Transit to Resident Ratio (RTR) 52
Modal split (Passenger) - by trips - Car 48
Modal split (Passenger) - by trips - Public Transport 50
Traffic Index 45
Average travel speed 43
Percentage Access to Public Transport 62
Share of population with access to open public spaces 54
Surface transport SOx Concentration 39
Surface transport NOx Concentration 39


In [466]:
urls = ["https://citydensity.com/api/v1/cities/1138958/density-profile", # kabul
"https://citydensity.com/api/v1/cities/2158177/density-profile", # melbourne
"https://citydensity.com/api/v1/cities/2147714/density-profile", # sydney
 "https://citydensity.com/api/v1/cities/1205733/density-profile", # chattogram
 "https://citydensity.com/api/v1/cities/1185241/density-profile", # dhaka
 "https://citydensity.com/api/v1/cities/1816670/density-profile", # beijing
 "https://citydensity.com/api/v1/cities/1815286/density-profile", # chengdu
 "https://citydensity.com/api/v1/cities/1814906/density-profile", # chongqing
 "https://citydensity.com/api/v1/cities/1809858/density-profile", # guangzhou
 "https://citydensity.com/api/v1/cities/1796236/density-profile", # shanghai
 "https://citydensity.com/api/v1/cities/1795565/density-profile", # shenzhen
 "https://citydensity.com/api/v1/cities/1790645/density-profile", # xiamen
 "https://citydensity.com/api/v1/cities/1819729/density-profile", # hong kong
 "https://citydensity.com/api/v1/cities/1277333/density-profile", # bengaluru
 "https://citydensity.com/api/v1/cities/1264527/density-profile", # chennai
 "https://citydensity.com/api/v1/cities/1273294/density-profile", # delhi
 'https://citydensity.com/api/v1/cities/1275004/density-profile', # kolkata
 "https://citydensity.com/api/v1/cities/1275339/density-profile", # mumbai
 "https://citydensity.com/api/v1/cities/1642911/density-profile", # jakarta
 "https://citydensity.com/api/v1/cities/112931/density-profile", # tehran
 "https://citydensity.com/api/v1/cities/1863967/density-profile", # fukuoka
 "https://citydensity.com/api/v1/cities/1853909/density-profile", # osaka
 "https://citydensity.com/api/v1/cities/2128295/density-profile", # sapporo
 "https://citydensity.com/api/v1/cities/1850147/density-profile", # tokyo
 "https://citydensity.com/api/v1/cities/1735161/density-profile", # kuala lumpur
 "https://citydensity.com/api/v1/cities/2193733/density-profile", # auckland
 "https://citydensity.com/api/v1/cities/1172451/density-profile", # lahore
 "https://citydensity.com/api/v1/cities/1701668/density-profile", # metro manila
 "https://citydensity.com/api/v1/cities/1838524/density-profile",  # busan
 "https://citydensity.com/api/v1/cities/1835848/density-profile", # seoul
 "https://citydensity.com/api/v1/cities/524901/density-profile",  # moscow
 "https://citydensity.com/api/v1/cities/498817/density-profile", # saint petersburg
 "https://citydensity.com/api/v1/cities/1880252/density-profile", # singapore
 "https://citydensity.com/api/v1/cities/1673820/density-profile", # kaohsiung
 "https://citydensity.com/api/v1/cities/1609350/density-profile", # bangkok
 "https://citydensity.com/api/v1/cities/1581130/density-profile", # hanoi
 "https://citydensity.com/api/v1/cities/1566083/density-profile", # ho chi minh
 "https://citydensity.com/api/v1/cities/3435910/density-profile", # buenos aires
 "https://citydensity.com/api/v1/cities/3448439/density-profile", # sao paulo
 "https://citydensity.com/api/v1/cities/6167865/density-profile", # toronto
 "https://citydensity.com/api/v1/cities/3688689/density-profile", # bogota
 "https://citydensity.com/api/v1/cities/360630/density-profile",  # cairo
 "https://citydensity.com/api/v1/cities/2988507/density-profile", # paris
 "https://citydensity.com/api/v1/cities/2950159/density-profile", # berlin
 "https://citydensity.com/api/v1/cities/98182/density-profile", # baghdad
 "https://citydensity.com/api/v1/cities/3173435/density-profile", # milan
 "https://citydensity.com/api/v1/cities/3530597/density-profile", # mexico city
 "https://citydensity.com/api/v1/cities/2759794/density-profile", # amsterdam
 "https://citydensity.com/api/v1/cities/2332459/density-profile", # lagos
 "https://citydensity.com/api/v1/cities/2267057/density-profile", # lisbon
 "https://citydensity.com/api/v1/cities/108410/density-profile", # riyadh
 "https://citydensity.com/api/v1/cities/993800/density-profile", # johanessburg
 "https://citydensity.com/api/v1/cities/3117735/density-profile", # madrid
 "https://citydensity.com/api/v1/cities/745044/density-profile", # istanbul
 "https://citydensity.com/api/v1/cities/292223/density-profile", # dubai
 "https://citydensity.com/api/v1/cities/2643743/density-profile", # london
 "https://citydensity.com/api/v1/cities/5368361/density-profile", # los angeles
 "https://citydensity.com/api/v1/cities/5128581/density-profile", # new york
 "https://citydensity.com/api/v1/cities/1274746/density-profile", # chandigarh
 "https://citydensity.com/api/v1/cities/1526384/density-profile", # almaty
 "https://citydensity.com/api/v1/cities/1298824/density-profile", # yangon
 'https://citydensity.com/api/v1/cities/1248991/density-profile', # colombo
 "https://citydensity.com/api/v1/cities/1668341/density-profile", # taipei
]

import requests
density = []

with requests.Session() as session:
    for url in urls:
        response = session.get(url)
        response.raise_for_status()

        data = response.json()
        density.append(data["metrics"][9]["density_cum_with_water"])

master_table['Population Density'] = density
master_table.to_csv('cleaned.csv', index=True)

In [467]:
print(master_table.index)

Index(['Kabul', 'Melbourne', 'Sydney', 'Chattogram', 'Dhaka', 'Beijing',
       'Chengdu', 'Chongqing', 'Guangzhou', 'Shanghai', 'Shenzhen', 'Xiamen',
       'Hong Kong', 'Bengaluru', 'Chennai', 'Delhi [New Delhi]', 'Kolkata',
       'Mumbai', 'Jakarta', 'Tehran', 'Fukuoka', 'Osaka [Kyoto]', 'Sapporo',
       'Tokyo', 'Kuala Lumpur', 'Auckland', 'Lahore', 'Metro Manila', 'Busan',
       'Seoul', 'Moscow', 'Saint Petersburg', 'Singapore', 'Kaohsiung',
       'Bangkok', 'Hanoi', 'Ho Chi Minh City', 'Buenos Aires', 'São Paulo',
       'Toronto', 'Bogota', 'Cairo', 'Paris', 'Berlin', 'Baghdad', 'Milan',
       'Mexico City', 'Amsterdam', 'Lagos', 'Lisbon', 'Riyadh', 'Johannesburg',
       'Madrid', 'Istanbul', 'Dubai', 'London', 'Los Angeles', 'New York',
       'Chandigarh', 'Almaty', 'Yangon', 'Colombo', 'New Taipei [Taipei]'],
      dtype='object', name='Urban Center')


In [468]:
print(master_table[['Population Density']])

                     Population Density
Urban Center                           
Kabul                        12921.8820
Melbourne                     3835.8025
Sydney                        4864.0645
Chattogram                    7598.5190
Dhaka                        18936.2230
...                                 ...
Chandigarh                    6426.0103
Almaty                        5419.0660
Yangon                       12263.2960
Colombo                       8613.1280
New Taipei [Taipei]          15701.5000

[63 rows x 1 columns]
